In [14]:
import json
import pandas as pd
from tqdm import tqdm
import pickle
import os
import datasets

/projects/iris/rferreir/.envs/spe/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [16]:
# Path the directory containing the extracted data
dir_path = '/projects/iris/rferreir/datasets/TourismQA/original_data'

# Path to the directory containing the HF datasets 
output_path = "/projects/iris/rferreir/hub_datasets"

In [3]:
def load_json(path):
    with open(path, 'r') as f:
        data = json.load(f)
    return data

In [4]:
cities = load_json(os.path.join(dir_path, 'final_cities_lat_long.json'))
knowledge_sum = load_json(os.path.join(dir_path, 'TourQue_Knowledge_SelSum.json'))
knowledge = load_json(os.path.join(dir_path, 'TourQue_Knowledge_Sel.json'))

In [5]:
map_city = {}
for city in cities:
    map_city[city['city_id']] = {"coord":(city['location']['lat'], city['location']['lng']), "name":city['city_name']}

In [6]:
print(len(knowledge_sum))
print(knowledge_sum['0_R_7327'])

114520
{'review': ['Verdict: A great place to grab a beer and a warm pretzel at the same time.\n', 'Pros: A great bar to sit down and have a beer in peace and quiet .\n', 'Cons: The bar may be too large for some people to comfortably sit at the end of the bar, but this is a testament to the quality of the staff and the fact that it is a good bar to get a drink at .\n'], 'lat_long': [40.7201861, -73.9846227], 'name': 'Donnybrook, 35 Clinton St, New York City, NY 10002-2426', 'cluster_map': {}}


In [7]:
print(len(knowledge))
print(knowledge['0_R_7327'])

114520
{'review': ['It was the only place that we didnt have on a list of places to go and I have to say it was one of the high lights of the night.', 'They were not serving food when we arrived, but the bar tender ordered a pizza for us which we ate at the bar :-) Definitely include it in an East Village pub crawl', 'Nice bar to grab a beer and a warm pretzel.', 'I wanted a nice pub to sit down and have a beer in peace and quiet.', 'You always find a seat in this place .', 'A great prerequisite to the delancey for the final blow out.', 'Good beer and good service .', 'They have Magners (just what you need on a hot NYC summer day) The pace is easy going, the staff are friendly and the drinks are reasonably priced (similar to Dublin).', 'Staff is kindle and guinness is a real guinness .', 'The music was R&B / HIPHOP / Pop and the whole place had a really good vibe.'], 'lat_long': [40.7201861, -73.9846227], 'name': 'Donnybrook, 35 Clinton St, New York City, NY 10002-2426', 'cluster_map':

In [8]:
def complete_data(data):
    new_data = []
    i = 0
    step = 100
    while i < len(data):
        q = data[i]
        city = map_city[int(q['city'])]
        q1 = {}
        q1["question"] = q['question']
        q1['city'] = city
        q1['tagged_locations'] = q['tagged_locations']
        q1['tagged_locations_lat_long'] = q['tagged_locations_lat_long']
        names = []
        adresses = []
        sum_reviews = []
        reviews = []
        lat_longs = []
        for idx in q['all_answer_entities']:
            if idx in knowledge_sum:
                k_sum = knowledge_sum[idx]
                k = knowledge[idx]
                split = k['name'].replace("[", "").replace("]", "").split(',')
                names.append(split[0])
                adresses.append(", ".join(split[1:]).strip())
                sum_reviews.append(' '.join(k_sum['review']))
                reviews.append(k['review'])
                if 'lat_long' in k:
                    lat_longs.append(k['lat_long'])
                else:
                    lat_longs.append("Unknown")
        q1['answers_names'] = names
        q1['answers_adresses'] = adresses
        q1['answers_sum_reviews'] = sum_reviews
        q1['answers_reviews'] = reviews
        q1['answers_lat_longs'] = lat_longs
                
        new_data.append(q1)
        i += len(q['all_answer_entities'])
    return new_data

In [9]:
test_data = load_json(os.path.join(dir_path, "test.json"))
test = complete_data(test_data)
print(test[0])

{'question': "Hi, everyone! We (I and my wife) will be seeing Les Misérables at Queens Theatre in London. As the play ends around at 22:00 (it's so late!), we are planning to have dinner before the show. We have to arrive at the theatre by 19:00. What restaurants do you recommend? We would like to have dinner near Queens Theatre. The budget is about ￡20-30 per person. We're looking forward to your kind and useful suggestions. P. S. Is it dangerous to walk around Queen's theatre after the show? In fact, we are living in Japan, which is a very secure country. We are not sure whether London is a safe city. When we travelled in New York, we were advised not to go out after midnight.", 'city': {'coord': (51.5073509, -0.1277583), 'name': 'London'}, 'tagged_locations': ['Queens_17 Theatre_18 in_19 London_20', 'Queens_70 Theatre_71', 'P._93 S._94', "Queen_101 's_102 theatre_103", 'the_52 theatre_53', 'Les_14 Misérables_15'], 'tagged_locations_lat_long': [[51.565460205078125, 0.2195200026035308

In [10]:
print(len(test))

2173


In [11]:
train_data = load_json(os.path.join(dir_path, "train.json"))
train = complete_data(train_data)
print(len(train))

19960


In [12]:
dev_data = load_json(os.path.join(dir_path, "valid.json"))
dev = complete_data(dev_data)
print(len(dev))

2119


## Converting to HF dataset

In [17]:
d_test = datasets.Dataset.from_list(test)
d_train = datasets.Dataset.from_list(train)
d_dev = datasets.Dataset.from_list(dev)

d1 = datasets.DatasetDict({
        "test": d_test,
        "train": d_train,
        "validation": d_dev
    })

print(d1)
print(d1['train'][0])

d1.save_to_disk(os.path.join(output_path, "TourismQA"))

DatasetDict({
    test: Dataset({
        features: ['question', 'city', 'tagged_locations', 'tagged_locations_lat_long', 'answers_names', 'answers_adresses', 'answers_sum_reviews', 'answers_reviews', 'answers_lat_longs'],
        num_rows: 2173
    })
    train: Dataset({
        features: ['question', 'city', 'tagged_locations', 'tagged_locations_lat_long', 'answers_names', 'answers_adresses', 'answers_sum_reviews', 'answers_reviews', 'answers_lat_longs'],
        num_rows: 19960
    })
    validation: Dataset({
        features: ['question', 'city', 'tagged_locations', 'tagged_locations_lat_long', 'answers_names', 'answers_adresses', 'answers_sum_reviews', 'answers_reviews', 'answers_lat_longs'],
        num_rows: 2119
    })
})
{'question': 'In three weeks I will be attending a conference at the Marriott downtown Mag Mile. We have a budge of $25 a day for food. We are all teachers. Any CHEAP suggestions for restaurants/café for the three days we will stay there. I like mainly Ameri

Saving the dataset (1/1 shards): 100%|██████████| 2119/2119 [00:00<00:00, 41683.77 examples/s]


In [19]:
import hashlib
import json

d2 = datasets.load_dataset("rfr2003/Geo_Benchmark", "TourismQA")
# We check that the content of the dataset is the same
def content_hash(ds):
    h = hashlib.sha256()
    for ex in ds:
        h.update(json.dumps(ex, sort_keys=True).encode())
    return h.hexdigest()

for split in ['train', 'test', 'validation']:
    assert content_hash(d1[split]) == content_hash(d2[split])

Generating validation split: 100%|██████████| 2119/2119 [00:00<00:00, 31431.51 examples/s]
